In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from openai import OpenAI

### Config

In [2]:
DATA_FILE = "../data/processed/cleaned_emotions_shared.csv"
SPLIT_FILE = "../data/splits/shared_split.json"

ML_METADATA_FILE = "../models/ml_all_models_metadata.json"
DL_MODEL_FILE = "../models/bilstm_model.keras"
DL_META_FILE = "../models/dl_metadata.json"
TOKENIZER_FILE = "../models/tokenizer.json"

OUTPUT_PRED_FILE = "../data/processed/hybrid_all_predictions.csv"
OUTPUT_SUMMARY_FILE = "../data/processed/hybrid_all_summary.csv"
OUTPUT_PER_CLASS_FILE = "../data/processed/hybrid_all_per_class.csv"

CONF_THRESHOLD = 0.70
USE_QWEN = True

### DEEPSEEK_MODEL via Ollama

In [ ]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

DEEPSEEK_MODEL = "deepseek-r1:8b"
print("AI ready:", DEEPSEEK_MODEL)

AI ready: deepseek-r1:8b


### QWEN FUNCTION

In [4]:
def qwen_refine(text, pred_label, allowed_labels):
    try:
        labels_str = ", ".join(allowed_labels)

        prompt = f"""You are an emotion classification assistant.

Text:
"{text}"

Predicted label: "{pred_label}"
Allowed labels: {labels_str}

Task:
Check whether the predicted label matches the meaning of the text.

Rules:
1. If the predicted label is correct, return it unchanged.
2. If the predicted label is incorrect, return a better label from the allowed labels.
3. Reply with ONLY one label.
"""

        response = client.chat.completions.create(
            model=DEEPSEEK_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = response.choices[0].message.content.strip().lower()

        for label in allowed_labels:
            if label.lower() == output:
                return label

        for label in allowed_labels:
            if label.lower() in output:
                return label

        return pred_label

    except Exception as e:
        print("Qwen error:", e)
        return pred_label

### Helper: metrics

In [5]:
def get_summary_metrics(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1_Macro": f1_score(y_true, y_pred, average="macro"),
        "F1_Weighted": f1_score(y_true, y_pred, average="weighted")
    }

def get_per_class_report(y_true, y_pred, model_name, labels):
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0
    )

    rows = []
    for label in labels:
        rows.append({
            "Model": model_name,
            "Emotion": label,
            "Precision": report[label]["precision"],
            "Recall": report[label]["recall"],
            "F1_Score": report[label]["f1-score"],
            "Support": int(report[label]["support"])
        })
    return pd.DataFrame(rows)

def print_model_result(y_true, y_pred, model_name, labels):
    print(f"\n===== {model_name} =====")
    print("Accuracy    :", accuracy_score(y_true, y_pred))
    print("F1 Macro    :", f1_score(y_true, y_pred, average="macro"))
    print("F1 Weighted :", f1_score(y_true, y_pred, average="weighted"))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred, labels=labels))

### Load data

In [6]:
df = pd.read_csv(DATA_FILE)

with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split = json.load(f)

test_df = df[df["row_id"].isin(split["test_ids"])].copy()

texts = test_df["text"].tolist()
y_true = test_df["label"].tolist()

results = []
summary_rows = []
per_class_frames = []

result_df = test_df[["row_id", "text", "label"]].copy()

### LOAD ML

In [7]:
with open(ML_METADATA_FILE, "r", encoding="utf-8") as f:
    ml_meta = json.load(f)

labels = ml_meta["labels"]
vectorizer = joblib.load(ml_meta["vectorizer_file"])
X_test_ml = vectorizer.transform(texts)

### ML LOOP

In [ ]:
for model_name, info in ml_meta.items():
    if model_name in ["labels", "vectorizer_file"]:
        continue

    print("\n===== ML:", model_name, "=====")

    model = joblib.load(info["model_path"])

    preds = model.predict(X_test_ml)

    # confidence from predict_proba
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_test_ml)
        confs = probs.max(axis=1)
    else:
        confs = np.ones(len(preds)) * 0.5

    base_col = info["prediction_column"]
    result_df[base_col] = preds
    result_df[base_col.replace("_pred", "_confidence")] = confs

    # baseline result
    print_model_result(y_true, preds, model_name, labels)
    summary_rows.append(get_summary_metrics(y_true, preds, model_name))
    per_class_frames.append(get_per_class_report(y_true, preds, model_name, labels))

    # hybrid result
    final_preds = []

    for text, pred, conf in zip(texts, preds, confs):
        if USE_QWEN and conf < CONF_THRESHOLD:
            new_pred = qwen_refine(text, pred, labels)
        else:
            new_pred = pred
        final_preds.append(new_pred)

    hybrid_name = f"{model_name} + Qwen"
    hybrid_col = base_col.replace("_pred", "_qwen_pred")
    result_df[hybrid_col] = final_preds

    print_model_result(y_true, final_preds, hybrid_name, labels)
    summary_rows.append(get_summary_metrics(y_true, final_preds, hybrid_name))
    per_class_frames.append(get_per_class_report(y_true, final_preds, hybrid_name, labels))


===== ML: Naive Bayes =====

===== Naive Bayes =====
Accuracy    : 0.8827160493827161
F1 Macro    : 0.8981922477981571
F1 Weighted : 0.8808117346422094

Classification Report:
              precision    recall  f1-score   support

       angry       0.96      1.00      0.98        49
     disgust       0.85      0.94      0.89        64
        fear       0.83      0.72      0.77        60
       happy       0.97      0.97      0.97        35
     neutral       0.81      0.80      0.81        75
         sad       0.98      0.98      0.98        41

    accuracy                           0.88       324
   macro avg       0.90      0.90      0.90       324
weighted avg       0.88      0.88      0.88       324

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 1 60  0  0  3  0]
 [ 0  6 43  0 10  1]
 [ 0  0  1 34  0  0]
 [ 1  5  8  1 60  0]
 [ 0  0  0  0  1 40]]


###  LOAD DL

In [ ]:
print("\n===== DL: BiLSTM =====")

dl_model = load_model(DL_MODEL_FILE)

with open(DL_META_FILE, "r", encoding="utf-8") as f:
    dl_meta = json.load(f)

with open(TOKENIZER_FILE, "r", encoding="utf-8") as f:
    tokenizer_json = f.read()

tokenizer = tokenizer_from_json(tokenizer_json)

max_len = dl_meta["max_sequence_length"]
dl_labels = dl_meta["classes"]

X_test_seq = tokenizer.texts_to_sequences(texts)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

# DL predict -> softmax probabilities
dl_probs = dl_model.predict(X_test_pad, verbose=0)
dl_preds_idx = np.argmax(dl_probs, axis=1)
dl_confs = dl_probs.max(axis=1)

dl_preds = [dl_labels[i] for i in dl_preds_idx]

result_df["dl_pred"] = dl_preds
result_df["dl_confidence"] = dl_confs

# baseline DL
print_model_result(y_true, dl_preds, "BiLSTM", labels)
summary_rows.append(get_summary_metrics(y_true, dl_preds, "BiLSTM"))
per_class_frames.append(get_per_class_report(y_true, dl_preds, "BiLSTM", labels))

# hybrid DL
final_dl_preds = []

for text, pred, conf in zip(texts, dl_preds, dl_confs):
    if USE_QWEN and conf < CONF_THRESHOLD:
        new_pred = qwen_refine(text, pred, labels)
    else:
        new_pred = pred
    final_dl_preds.append(new_pred)

result_df["dl_qwen_pred"] = final_dl_preds

print_model_result(y_true, final_dl_preds, "BiLSTM + Qwen", labels)
summary_rows.append(get_summary_metrics(y_true, final_dl_preds, "BiLSTM + Qwen"))
per_class_frames.append(get_per_class_report(y_true, final_dl_preds, "BiLSTM + Qwen", labels))


===== DL: BiLSTM =====

===== BiLSTM =====
Accuracy    : 0.9012345679012346
F1 Macro    : 0.9132275733292298
F1 Weighted : 0.8983912465743429

Classification Report:
              precision    recall  f1-score   support

       angry       0.96      1.00      0.98        49
     disgust       0.89      0.97      0.93        64
        fear       0.89      0.68      0.77        60
       happy       1.00      0.97      0.99        35
     neutral       0.81      0.87      0.84        75
         sad       0.95      1.00      0.98        41

    accuracy                           0.90       324
   macro avg       0.92      0.92      0.91       324
weighted avg       0.90      0.90      0.90       324

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 62  1  0  1  0]
 [ 1  5 41  0 13  0]
 [ 0  0  0 34  1  0]
 [ 1  3  4  0 65  2]
 [ 0  0  0  0  0 41]]

===== BiLSTM + Qwen =====
Accuracy    : 0.9012345679012346
F1 Macro    : 0.9053646561351569
F1 Weighted : 0.8971512419814465

Classification Rep

### Final tables

In [ ]:
summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["Accuracy", "F1_Macro"],
    ascending=False
).reset_index(drop=True)

per_class_df = pd.concat(per_class_frames, ignore_index=True)

print("\n===== FINAL SUMMARY =====")
print(summary_df)


===== FINAL SUMMARY =====
                           Model  Accuracy  F1_Macro  F1_Weighted
0  Support Vector Machine + Qwen  0.910494  0.917098     0.909107
1         Support Vector Machine  0.904321  0.919068     0.903556
2                         BiLSTM  0.901235  0.913228     0.898391
3                  BiLSTM + Qwen  0.901235  0.905365     0.897151
4            Logistic Regression  0.895062  0.912052     0.894449
5                    Naive Bayes  0.882716  0.898192     0.880812
6             Naive Bayes + Qwen  0.870370  0.878272     0.871451
7           Random Forest + Qwen  0.864198  0.873149     0.863785
8                  Random Forest  0.861111  0.885660     0.863335
9     Logistic Regression + Qwen  0.842593  0.849000     0.844944


### Save

In [ ]:
result_df.to_csv(OUTPUT_PRED_FILE, index=False, encoding="utf-8-sig")
summary_df.to_csv(OUTPUT_SUMMARY_FILE, index=False, encoding="utf-8-sig")
per_class_df.to_csv(OUTPUT_PER_CLASS_FILE, index=False, encoding="utf-8-sig")

print("\nSaved:")
print("-", OUTPUT_PRED_FILE)
print("-", OUTPUT_SUMMARY_FILE)
print("-", OUTPUT_PER_CLASS_FILE)
print("Done")


Saved:
- ../data/processed/hybrid_all_predictions.csv
- ../data/processed/hybrid_all_summary.csv
- ../data/processed/hybrid_all_per_class.csv
Done
